In [24]:
import json
import math
from pathlib import Path

import numpy as np
import yaml
from sympy.physics.quantum.cg import CG

EPS = 1e-8

# Isolated TF-PWA amplitude flow for all configured components

This notebook retains the validated $\psi(4040)$ execution flow and extends the isolated NumPy implementation to all 13 active chains in `config_a.yml`. TF-PWA is imported only for configured event generation and final reference comparisons.


# Functions

In [25]:
# -----------------------------------------------------------------------------
# Common, self-contained copies/adaptations of the TF-PWA amplitude math.
# Psi(4040) chain. These functions intentionally do not import tf_pwa.
# -----------------------------------------------------------------------------


def dot3(a, b):
    return np.sum(a * b, axis=-1)


def norm3(a):
    return np.linalg.norm(a, axis=-1)


def unit(v):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / n


def cross_unit(a, b):
    # Copied from TF-PWA Vector3.cross_unit in spirit: if the cross product is
    # degenerate, bias the second vector before normalizing.
    cro = np.cross(a, b)
    norm = np.linalg.norm(cro, axis=-1, keepdims=True)
    mask = norm < EPS
    bias_other = np.ones_like(norm) + b
    cro = np.where(mask, np.cross(a, bias_other), cro)
    return unit(cro)


def angle_from(v, x_axis, y_axis):
    # Angle of vector v measured in the coordinate basis (x_axis, y_axis).
    return np.arctan2(dot3(v, y_axis), dot3(v, x_axis))


def angle_zx_z_getx(z1, x1, z2):
    # Adapted from TF-PWA EulerAngle.angle_zx_z_getx.
    # Physical meaning: construct the Euler rotation from an old frame (z1,x1)
    # to a new helicity frame whose z-axis follows a daughter momentum z2.
    u_z1 = unit(z1)
    u_z2 = unit(z2)
    u_y1 = cross_unit(z1, x1)
    u_x1 = cross_unit(u_y1, z1)
    u_yr = cross_unit(z1, z2)
    u_xr = cross_unit(u_yr, z1)
    alpha = angle_from(u_xr, u_x1, u_y1)
    beta = angle_from(u_z2, u_z1, u_xr)
    gamma = np.zeros_like(beta)
    u_x2 = cross_unit(u_yr, u_z2)
    return {"alpha": alpha, "beta": beta, "gamma": gamma}, u_x2


def invariant_mass(p4):
    # TF-PWA LorentzVector.M with metric (+,-,-,-).
    m2 = p4[..., 0] ** 2 - np.sum(p4[..., 1:] ** 2, axis=-1)
    return np.sqrt(np.abs(m2))


def boost_vector(p4):
    return p4[..., 1:] / p4[..., 0:1]


def boost(p4, beta):
    # Adapted from TF-PWA LorentzVector.boost.
    beta2 = np.sum(beta * beta, axis=-1)
    gamma = 1.0 / np.sqrt(1.0 - beta2)
    bp = np.sum(beta * p4[..., 1:], axis=-1)
    gamma2 = np.where(beta2 > EPS, (gamma - 1.0) / beta2, 0.0)
    spatial = p4[..., 1:]
    spatial = spatial + gamma2[..., None] * bp[..., None] * beta
    spatial = spatial + gamma[..., None] * p4[..., 0:1] * beta
    energy = (gamma * (p4[..., 0] + bp))[..., None]
    return np.concatenate([energy, spatial], axis=-1)


def rest_vector(core_p4, other_p4):
    # Boost other_p4 into the rest frame of core_p4.
    return boost(other_p4, -boost_vector(core_p4))


def get_relative_p2(m0, m1, m2):
    # Two-body breakup momentum squared, copied from TF-PWA formula.get_relative_p2.
    return ((m0 * m0 - (m1 + m2) ** 2) * (m0 * m0 - (m1 - m2) ** 2)) / (4 * m0 * m0)


def bprime_polynomial(l, z):
    # Blatt-Weisskopf polynomials used by TF-PWA up to the orders needed here.
    coeff = {
        0: [1.0],
        1: [1.0, 1.0],
        2: [1.0, 3.0, 9.0],
        3: [1.0, 6.0, 45.0, 225.0],
    }
    return np.polyval(coeff[int(l)], z)


def bprime_q2(l, q2, q02, d=3.0):
    # Adapted from TF-PWA breit_wigner.Bprime_q2.
    z0 = q02 * d**2
    z = q2 * d**2
    ratio = bprime_polynomial(l, z0) / bprime_polynomial(l, z)
    return np.sqrt(np.where(ratio > 0, ratio, 1.0))


def barrier_factor2(l, mass, q2, q02, d=3.0, barrier_factor_norm=True):
    # Adapted from HelicityDecay.get_barrier_factor2 for this chain.
    # Physical meaning: centrifugal-barrier factor q^L B'_L(q,q0,d), normalized by q0^L.
    tmp = q2 ** (l / 2) * bprime_q2(l, q2, q02, d)
    if barrier_factor_norm:
        tmp = tmp / np.abs(q02) ** (l / 2)
    return tmp.reshape(-1, 1)


def gamma_running(m, gamma0, q, q0, l, m0, d=3.0):
    # Adapted from TF-PWA breit_wigner.Gamma.
    qq0 = np.where(q0 > 1e-15, (q / q0) ** (2 * l + 1), 1.0)
    mm0 = m0 / m
    bp = (np.sqrt(bprime_polynomial(l, (q0 * d) ** 2)) / np.sqrt(bprime_polynomial(l, (q * d) ** 2))) ** 2
    return gamma0 * qq0 * mm0 * bp


def bwr(m, m0, gamma0, q, q0, l, d=3.0):
    # Adapted from TF-PWA breit_wigner.BWR.
    # Physical meaning: relativistic Breit-Wigner propagator with running width.
    gamma_m = gamma_running(m, gamma0, q, q0, l, m0, d)
    x = m0 * m0 - m * m
    y = m0 * gamma_m
    denom = x * x + y * y
    return x / denom + 1j * y / denom


def small_d_weight(j2):
    # Copied/adapted from TF-PWA dfun.small_d_weight. j2 means 2*j.
    ret = np.zeros((j2 + 1, j2 + 1, j2 + 1))

    def half_factorial(x):
        return math.factorial(x >> 1)

    for m in range(-j2, j2 + 1, 2):
        for n in range(-j2, j2 + 1, 2):
            for k in range(max(0, n - m), min(j2 - m, j2 + n) + 1, 2):
                ell = (2 * k + (m - n)) // 2
                sign = (-1) ** ((k + m - n) // 2)
                val = sign * math.sqrt(
                    half_factorial(j2 + m)
                    * half_factorial(j2 - m)
                    * half_factorial(j2 + n)
                    * half_factorial(j2 - n)
                )
                val /= (
                    half_factorial(j2 - m - k)
                    * half_factorial(j2 + n - k)
                    * half_factorial(k + m - n)
                    * half_factorial(k)
                )
                ret[ell][(m + j2) // 2][(n + j2) // 2] = val
    return ret


def small_d_matrix(theta, j2):
    theta = np.asarray(theta)
    powers = np.arange(0, j2 + 1).reshape(1, -1)
    half_theta = 0.5 * theta.reshape(-1, 1)
    sc = (np.sin(half_theta) ** powers) * (np.cos(half_theta) ** (j2 - powers))
    weights = small_d_weight(j2).reshape(j2 + 1, (j2 + 1) * (j2 + 1))
    return (sc @ weights).reshape(-1, j2 + 1, j2 + 1)


def d_matrix_conj(alpha, beta, gamma, j2):
    # Adapted from TF-PWA dfun.D_matrix_conj.
    # Physical meaning: conjugated Wigner D matrix D^J*(alpha,beta,gamma).
    m = np.arange(-j2 / 2, j2 / 2 + 1, 1).reshape(1, -1)
    d_small = small_d_matrix(beta, j2)
    exp_alpha = np.exp(1j * alpha.reshape(-1, 1) * m).reshape(-1, j2 + 1, 1)
    exp_gamma = np.exp(1j * gamma.reshape(-1, 1) * m).reshape(-1, 1, j2 + 1)
    return exp_alpha * exp_gamma * d_small.astype(complex)


def dfun_delta_v2(d, ja, la, lb, lc=(0,)):
    # Adapted from TF-PWA dfun.Dfun_delta_v2.
    ln = int(2 * ja + 1 + 0.1)
    idx = []
    max_idx = ln * ln
    for la_i in la:
        for lb_i in lb:
            for lc_i in lc:
                delta = lb_i - lc_i
                if abs(delta) <= ja:
                    idx.append(int((la_i + ja) * ln + delta + ja + 0.1))
                else:
                    idx.append(max_idx)
    flat = d.reshape(-1, ln * ln)
    padded = np.pad(flat, ((0, 0), (0, 1)))
    return padded[:, idx].reshape(-1, len(la), len(lb), len(lc))


def get_d_matrix_lambda(angle, ja, la, lb, lc=None):
    # Adapted from TF-PWA dfun.get_D_matrix_lambda.
    d = d_matrix_conj(angle["alpha"], angle["beta"], angle.get("gamma", np.zeros_like(angle["beta"])), int(2 * ja + 0.1))
    if lc is None:
        return dfun_delta_v2(d, ja, la, lb, (0,)).reshape(-1, len(la), len(lb))
    return dfun_delta_v2(d, ja, la, lb, lc)


def cg_coef(j1, j2, m1, m2, j, m):
    # Same convention as TF-PWA cg.cg_coef when SymPy is available.
    return float(CG(j1, m1, j2, m2, j, m).doit().evalf())


def cg_matrix(ja, jb, jc, ls_list, out_spins):
    # Adapted from HelicityDecay._get_cg_matrix.
    # Physical meaning: LS-to-helicity transformation matrix.
    ret = np.zeros((len(ls_list), len(out_spins[0]), len(out_spins[1])))
    for i, (ell, spin) in enumerate(ls_list):
        for ib, lambda_b in enumerate(out_spins[0]):
            for ic, lambda_c in enumerate(out_spins[1]):
                ret[i, ib, ic] = (
                    math.sqrt(2 * ell + 1)
                    / math.sqrt(2 * ja + 1)
                    * cg_coef(jb, jc, lambda_b, -lambda_c, spin, lambda_b - lambda_c)
                    * cg_coef(ell, spin, 0, lambda_b - lambda_c, ja, lambda_b - lambda_c)
                )
    return ret


def helicity_decay_amp(core_j, out_js, core_spins, out_spins, ls_list, g_ls, angle, mass_core, q2, q02, has_barrier=True, barrier_norm=True):
    # Adapted from HelicityDecay.get_amp -> get_helicity_amp -> get_ls_amp -> get_D_matrix_term.
    # Physical meaning: tensor amplitude for one two-body decay vertex.
    if has_barrier:
        ell = ls_list[0][0]
        bf = barrier_factor2(ell, mass_core, q2, q02, d=3.0, barrier_factor_norm=barrier_norm)
        m_dep = np.array([[g_ls]], dtype=complex) * bf.astype(complex)
    else:
        m_dep = np.array([[g_ls]], dtype=complex)

    cg = cg_matrix(core_j, out_js[0], out_js[1], ls_list, out_spins).astype(complex)
    h = np.sum(m_dep.reshape(-1, len(ls_list), 1, 1) * cg.reshape(len(ls_list), len(out_spins[0]), len(out_spins[1])), axis=1)
    h = h.reshape(-1, 1, len(out_spins[0]), len(out_spins[1]))
    d_conj = get_d_matrix_lambda(angle, core_j, core_spins, out_spins[0], out_spins[1])
    return h * d_conj.reshape(-1, len(core_spins), len(out_spins[0]), len(out_spins[1]))


def compute_chain_boosts(particle_p4, chain):
    # Adapted from TF-PWA cal_chain_boost.
    # Physical meaning: boost daughters step-by-step into each mother rest frame.
    particle_set = {name for _, outs in chain for name in outs}
    core_decay_map = {}
    part_data = {}
    pending = list(chain)
    while pending:
        extra = []
        for core, outs in pending:
            if core == "Bp":
                p_rest = particle_p4[core]
                part_data[core] = {"rest_p": {}}
                for out in outs:
                    core_decay_map[out] = core
                    part_data[core]["rest_p"][out] = rest_vector(p_rest, particle_p4[out])
                    particle_set.discard(out)
                for other in list(particle_set):
                    part_data[core]["rest_p"][other] = rest_vector(p_rest, particle_p4[other])
            elif core in core_decay_map:
                parent = core_decay_map[core]
                p_rest = part_data[parent]["rest_p"][core]
                part_data[core] = {"rest_p": {}}
                for out in outs:
                    core_decay_map[out] = core
                    part_data[core]["rest_p"][out] = rest_vector(p_rest, part_data[parent]["rest_p"][out])
                    particle_set.discard(out)
                for other in list(particle_set):
                    part_data[core]["rest_p"][other] = rest_vector(p_rest, part_data[parent]["rest_p"][other])
            else:
                extra.append((core, outs))
        pending = extra
    return part_data


def calculate_helicity_angles(particle_p4, chain):
    # Adapted from TF-PWA cal_helicity_angle for this chain.
    # Physical meaning: construct the helicity angles used in Wigner-D functions.
    part_data = compute_chain_boosts(particle_p4, chain)
    set_x = {"Bp": np.array([[1.0, 0.0, 0.0]])}
    set_z = {"Bp": np.array([[0.0, 0.0, 1.0]])}
    angles = {}
    for core, outs in chain:
        angles[core] = {}
        bias = -np.pi
        for out in outs:
            z2 = part_data[core]["rest_p"][out][..., 1:]
            ang, x_axis = angle_zx_z_getx(set_z[core], set_x[core], z2)
            set_x[out] = x_axis
            set_z[out] = z2
            ang["alpha"] = (ang["alpha"] - bias) % (2 * np.pi) + bias
            bias -= np.pi
            angles[core][out] = ang
    return angles


def format_complex(z):
    z = np.asarray(z).reshape(-1)[0]
    sign = "+" if z.imag >= 0 else "-"
    return f"{z.real:.16g} {sign} {abs(z.imag):.16g}j"

In [26]:
# -----------------------------------------------------------------------------
# Additional isolated TF-PWA functions required by the full configured model.
# BWR_LS follows ParticleBWRLS and ParticleDecayLS from tf_pwa.amp.split_ls;
# the remaining helpers generalize the single-LS vertex calculation above.
# -----------------------------------------------------------------------------

def param_complex(base):
    return float(params[base + "r"]) * np.exp(1j * float(params[base + "i"]))


def spin_values(j):
    return tuple(np.arange(-j, j + 1, 1, dtype=float))


def ad_hoc_mass(m0, m_max, m_min):
    k = (m_max - m_min) / 2.0
    return k * (1.0 + np.tanh((2.0 * m0 - (m_max + m_min)) / k / 4.0)) + m_min


def event_relative_p2(m0, m1, m2):
    # Exact operation order of tf_pwa.cal_angle.Getp2, used for cached event q^2.
    mass_sum = m1 + m2
    mass_difference = m1 - m2
    product = (m0 - mass_sum) * (m0 + mass_sum)
    product *= (m0 - mass_difference) * (m0 + mass_difference)
    return product / (4.0 * m0 * m0)


def nominal_relative_p2(m0, m1, m2):
    # Exact operation order of HelicityDecay.get_relative_momentum2 for q0^2.
    mass_sum = m1 + m2
    mass_difference = m1 - m2
    product = (m0 - mass_sum) * (m0 + mass_sum)
    product *= (m0 - mass_difference) * (m0 + mass_difference)
    return product / (2.0 * m0) ** 2


def standard_vertex_mdep(ls_list, g_ls, mass_core, q2, q02, has_barrier=True):
    g_ls = np.asarray(g_ls, dtype=complex)
    n_events = np.asarray(mass_core).reshape(-1).shape[0]
    if not has_barrier:
        return np.broadcast_to(g_ls.reshape(1, -1), (n_events, len(g_ls))).copy()
    factors = [
        barrier_factor2(ell, mass_core, q2, q02, d=3.0, barrier_factor_norm=True).reshape(-1)
        for ell, _ in ls_list
    ]
    return np.stack(factors, axis=-1).astype(complex) * g_ls.reshape(1, -1)


def bwr_ls_vertex_mdep(m, m0, width, theta0, ls_list, q2, q02, g_ls, sign):
    fractions = np.asarray([np.cos(theta0), np.sin(theta0)], dtype=float)
    partial = []
    for fraction, (ell, _) in zip(fractions, ls_list):
        bf = np.sqrt(q2 / q02) ** ell * bprime_q2(ell, q2, q02, d=3.0)
        partial.append(fraction * bf)
    partial = np.stack(partial, axis=-1)
    a = m0 * m0 - m * m
    b = m0 * width * np.sqrt(q2 / q02) * np.sum(partial * partial, axis=-1) * m / m0
    denominator = a - 1j * b
    return sign * partial / denominator[:, None] * np.asarray(g_ls, dtype=complex).reshape(1, -1)


def helicity_decay_amp_from_mdep(core_j, out_js, core_spins, out_spins, ls_list, m_dep, angle):
    m_dep = np.asarray(m_dep, dtype=complex)
    cg = cg_matrix(core_j, out_js[0], out_js[1], ls_list, out_spins).astype(complex)
    h = np.sum(m_dep[:, :, None, None] * cg[None, :, :, :], axis=1)
    d_conj = get_d_matrix_lambda(angle, core_j, core_spins, out_spins[0], out_spins[1])
    return h[:, None, :, :] * d_conj.reshape(-1, len(core_spins), len(out_spins[0]), len(out_spins[1]))


def fitted_ls(prefix, count):
    return [param_complex(f"{prefix}_g_ls_{index}") for index in range(count)]


def dstd_component_amplitude(final_p4, spec):
    name = spec["name"]
    p4 = {key: np.asarray(value) for key, value in final_p4.items()}
    p4["Dst"] = p4["D0"] + p4["pi"]
    p4["R"] = p4["Dst"] + p4["D"]
    p4["Bp"] = p4["R"] + p4["K"]
    masses = {key: invariant_mass(value) for key, value in p4.items()}
    chain = [("Bp", ["R", "K"]), ("R", ["Dst", "D"]), ("Dst", ["D0", "pi"])]
    angles = calculate_helicity_angles(p4, chain)

    q2_root = event_relative_p2(masses["Bp"], masses["R"], masses["K"])
    q02_root = nominal_relative_p2(nominal_mass["Bp"], spec["mass"], nominal_mass["K"])
    root_prefix = f"Bp->{name}.K"
    root_g = fitted_ls(root_prefix, len(spec["root_ls"]))
    root_mdep = standard_vertex_mdep(spec["root_ls"], root_g, masses["Bp"], q2_root, q02_root)

    q2_decay = event_relative_p2(masses["R"], masses["Dst"], masses["D"])
    q0_mass = spec["mass"]
    if spec.get("below", False):
        q0_mass = ad_hoc_mass(
            q0_mass,
            nominal_mass["Bp"] - nominal_mass["K"],
            nominal_mass["Dst"] + nominal_mass["D"],
        )
    q02_decay = nominal_relative_p2(q0_mass, nominal_mass["Dst"], nominal_mass["D"])
    decay_prefix = f"{name}->Dst.D"
    decay_g = fitted_ls(decay_prefix, len(spec["decay_ls"]))
    if spec["model"] == "BWR_LS":
        decay_mdep = bwr_ls_vertex_mdep(
            masses["R"], spec["mass"], spec["width"], float(params[name + "_theta0"]),
            spec["decay_ls"], q2_decay, q02_decay, decay_g, spec["sign"],
        )
        particle_factor = np.ones_like(masses["R"], dtype=complex)
    else:
        decay_mdep = standard_vertex_mdep(spec["decay_ls"], decay_g, masses["R"], q2_decay, q02_decay)
        if spec["model"] == "BWR":
            particle_factor = spec["sign"] * bwr(
                masses["R"], spec["mass"], spec["width"],
                np.sqrt(q2_decay), np.sqrt(q02_decay), spec["decay_ls"][0][0], d=3.0,
            )
        elif spec["model"] == "New":
            alpha = float(params[name + "_alpha"])
            beta = float(params[name + "_beta"])
            particle_factor = spec["sign"] * np.exp(-(alpha + 1j * beta) * (masses["R"] ** 2 - spec["mass"] ** 2))
        else:
            particle_factor = np.full_like(masses["R"], spec["sign"], dtype=complex)

    dst_mdep = standard_vertex_mdep(((1, 0),), [param_complex("Dst->D0.pi_g_ls_0")], masses["Dst"], np.zeros_like(masses["Dst"]), 0.0, has_barrier=False)
    root_amp = helicity_decay_amp_from_mdep(0, (spec["j"], 0), (0,), (spin_values(spec["j"]), (0,)), spec["root_ls"], root_mdep, angles["Bp"]["R"])
    decay_amp = helicity_decay_amp_from_mdep(spec["j"], (1, 0), spin_values(spec["j"]), (spin_values(1), (0,)), spec["decay_ls"], decay_mdep, angles["R"]["Dst"])
    dst_amp = helicity_decay_amp_from_mdep(1, (0, 0), spin_values(1), ((0,), (0,)), ((1, 0),), dst_mdep, angles["Dst"]["D0"])
    total = param_complex(f"Bp->{name}.K{name}->Dst.DDst->D0.pi_total_0") * particle_factor
    tensor = np.einsum("...agd,...gfb,...fce,...->...abcde", root_amp, decay_amp, dst_amp, total)
    return tensor.reshape(len(masses["Bp"]), -1)[:, 0]


def dk_component_amplitude(final_p4, spec):
    name = spec["name"]
    p4 = {key: np.asarray(value) for key, value in final_p4.items()}
    p4["Dst"] = p4["D0"] + p4["pi"]
    p4["R"] = p4["D"] + p4["K"]
    p4["Bp"] = p4["R"] + p4["Dst"]
    masses = {key: invariant_mass(value) for key, value in p4.items()}
    chain = [("Bp", ["R", "Dst"]), ("R", ["D", "K"]), ("Dst", ["D0", "pi"])]
    angles = calculate_helicity_angles(p4, chain)

    q2_root = event_relative_p2(masses["Bp"], masses["R"], masses["Dst"])
    q02_root = nominal_relative_p2(nominal_mass["Bp"], spec["mass"], nominal_mass["Dst"])
    root_g = fitted_ls(f"Bp->{name}.Dst", len(spec["root_ls"]))
    root_mdep = standard_vertex_mdep(spec["root_ls"], root_g, masses["Bp"], q2_root, q02_root)

    q2_decay = event_relative_p2(masses["R"], masses["D"], masses["K"])
    q02_decay = nominal_relative_p2(spec["mass"], nominal_mass["D"], nominal_mass["K"])
    decay_g = fitted_ls(f"{name}->D.K", len(spec["decay_ls"]))
    decay_mdep = standard_vertex_mdep(spec["decay_ls"], decay_g, masses["R"], q2_decay, q02_decay)
    particle_factor = bwr(
        masses["R"], spec["mass"], spec["width"],
        np.sqrt(q2_decay), np.sqrt(q02_decay), spec["decay_ls"][0][0], d=3.0,
    )
    dst_mdep = standard_vertex_mdep(((1, 0),), [param_complex("Dst->D0.pi_g_ls_0")], masses["Dst"], np.zeros_like(masses["Dst"]), 0.0, has_barrier=False)

    root_amp = helicity_decay_amp_from_mdep(0, (spec["j"], 1), (0,), (spin_values(spec["j"]), spin_values(1)), spec["root_ls"], root_mdep, angles["Bp"]["R"])
    decay_amp = helicity_decay_amp_from_mdep(spec["j"], (0, 0), spin_values(spec["j"]), ((0,), (0,)), spec["decay_ls"], decay_mdep, angles["R"]["D"])
    dst_amp = helicity_decay_amp_from_mdep(1, (0, 0), spin_values(1), ((0,), (0,)), ((1, 0),), dst_mdep, angles["Dst"]["D0"])
    total = param_complex(f"Bp->{name}.Dst{name}->D.KDst->D0.pi_total_0") * particle_factor
    tensor = np.einsum("...agf,...gde,...fbc,...->...abcde", root_amp, decay_amp, dst_amp, total)
    return tensor.reshape(len(masses["Bp"]), -1)[:, 0]


def isolated_component_amplitude(final_p4, spec):
    if spec["topology"] == "dstd":
        return dstd_component_amplitude(final_p4, spec)
    return dk_component_amplitude(final_p4, spec)


# Load Input

In [27]:
# -----------------------------------------------------------------------------
# Leaf inputs: files plus the model parameters needed for the sampled-event test.
# -----------------------------------------------------------------------------

analysis_dir = Path.cwd()
if not (analysis_dir / "config_a.yml").exists():
    analysis_dir = analysis_dir / "Analysis"
print(f"Notebook working directory: {analysis_dir}")

with open(analysis_dir / "config_a.yml", "r", encoding="utf-8") as f:
    config_yml = yaml.safe_load(f)

with open(analysis_dir / "Resonances.yml", "r", encoding="utf-8") as f:
    resonances_yml = yaml.safe_load(f)

with open(analysis_dir / "final_params_full.json", "r", encoding="utf-8") as f:
    params = json.load(f)["value"]

print("Loaded config_a.yml, Resonances.yml, and final_params_full.json")

# Nominal masses from the YAML model plus Psi(4040) mass/width from the JSON parameters.
nominal_mass = {
    "Bp": float(config_yml["particle"]["$top"]["Bp"]["mass"]),
    "D": float(config_yml["particle"]["$finals"]["D"]["mass"]),
    "K": float(config_yml["particle"]["$finals"]["K"]["mass"]),
    "D0": float(config_yml["particle"]["$finals"]["D0"]["mass"]),
    "pi": float(config_yml["particle"]["$finals"]["pi"]["mass"]),
    "Dst": float(config_yml["particle"]["Dst"]["mass"]),
    "Psi(4040)": float(params["Psi(4040)_mass"]),
}
psi_width = float(params["Psi(4040)_width"])

# The script overwrites all selected Psi(4040) couplings to 1 + 0j for this isolated probe.
g_total = 1.0 + 0.0j
g_bp_to_psi_k = 1.0 + 0.0j
g_psi_to_dst_d = 1.0 + 0.0j
g_dst_to_d0_pi = 1.0 + 0.0j

print("Nominal masses:")
for name in ["Bp", "D", "K", "D0", "pi", "Dst", "Psi(4040)"]:
    print(f"  {name:>9}: {nominal_mass[name]}")
print(f"Psi(4040) width: {psi_width}")


Notebook working directory: c:\Users\gamma\Documents\Playground\B2DxDK.jl_fresh\Analysis
Loaded config_a.yml, Resonances.yml, and final_params_full.json
Nominal masses:
         Bp: 5.27934
          D: 1.86965
          K: 0.493677
         D0: 1.86483
         pi: 0.13957039
        Dst: 2.01026
  Psi(4040): 4.039
Psi(4040) width: 0.08


In [28]:
component_specs = [
    dict(name="X(3872)", topology="dstd", j=1, root_ls=((1, 1),), decay_ls=((0, 1), (2, 1)), model="BWR_LS", sign=-1.0, below=True),
    dict(name="X(3915)(0-)", topology="dstd", j=0, root_ls=((0, 0),), decay_ls=((1, 1),), model="BWR", sign=-1.0),
    dict(name="chi(c2)(3930)", topology="dstd", j=2, root_ls=((2, 2),), decay_ls=((2, 1),), model="BWR", sign=-1.0),
    dict(name="X(3940)(1.)", topology="dstd", j=1, root_ls=((1, 1),), decay_ls=((0, 1), (2, 1)), model="BWR_LS", sign=1.0),
    dict(name="X(3993)", topology="dstd", j=1, root_ls=((1, 1),), decay_ls=((0, 1), (2, 1)), model="BWR_LS", sign=-1.0),
    dict(name="Psi(4040)", topology="dstd", j=1, root_ls=((1, 1),), decay_ls=((1, 1),), model="BWR", sign=1.0),
    dict(name="X(4300)", topology="dstd", j=1, root_ls=((1, 1),), decay_ls=((0, 1), (2, 1)), model="BWR_LS", sign=1.0),
    dict(name="NR(0-)SPp", topology="dstd", j=0, root_ls=((0, 0),), decay_ls=((1, 1),), model="New", sign=-1.0),
    dict(name="NR(1.)PSp", topology="dstd", j=1, root_ls=((1, 1),), decay_ls=((0, 1),), model="one", sign=-1.0),
    dict(name="NR(0-)SPm", topology="dstd", j=0, root_ls=((0, 0),), decay_ls=((1, 1),), model="one", sign=1.0),
    dict(name="NR(1-)PPm", topology="dstd", j=1, root_ls=((1, 1),), decay_ls=((1, 1),), model="one", sign=1.0),
    dict(name="X0(2900)", topology="dk", j=0, root_ls=((1, 1),), decay_ls=((0, 0),), model="BWR", sign=1.0),
    dict(name="X1(2900)", topology="dk", j=1, root_ls=((0, 0), (1, 1), (2, 2)), decay_ls=((1, 0),), model="BWR", sign=1.0),
]

for spec in component_specs:
    spec["mass"] = float(params.get(spec["name"] + "_mass", 4.35))
    if spec["model"] in ("BWR", "BWR_LS"):
        spec["width"] = float(params[spec["name"] + "_width"])



print("Active configured components:")
for index, spec in enumerate(component_specs):
    print(f"  {index:2d}: {spec['name']:<18} {spec['topology']:<4} {spec['model']}")


Active configured components:
   0: X(3872)            dstd BWR_LS
   1: X(3915)(0-)        dstd BWR
   2: chi(c2)(3930)      dstd BWR
   3: X(3940)(1.)        dstd BWR_LS
   4: X(3993)            dstd BWR_LS
   5: Psi(4040)          dstd BWR
   6: X(4300)            dstd BWR_LS
   7: NR(0-)SPp          dstd New
   8: NR(1.)PSp          dstd one
   9: NR(0-)SPm          dstd one
  10: NR(1-)PPm          dstd one
  11: X0(2900)           dk   BWR
  12: X1(2900)           dk   BWR


# Random TF-PWA Event Test

This section samples one random TF-PWA phase-space event and reruns Steps 1-8 with the isolated functions to test event-by-event consistency.


In [29]:
print("Random event setup: generate one TF-PWA phase-space event for Bp -> D K D0 pi.")
import os
import sys
repo_root = analysis_dir.parent
tfpwa_src = repo_root / "tf-pwa"
if str(tfpwa_src) not in sys.path:
    sys.path.insert(0, str(tfpwa_src))

os.chdir(analysis_dir)
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import tensorflow as tf
import extra_amp
from tf_pwa.config_loader import ConfigLoader

sample_seed = int(np.random.default_rng().integers(0, 2**31 - 1))
print(f"  TF random seed: {sample_seed}")
np.random.seed(sample_seed)
tf.random.set_seed(sample_seed)
sample_generator_config = ConfigLoader("config_a.yml")
sample_phase_space = sample_generator_config.generate_phsp_p(1)
sampled_p4 = {particle.name: np.asarray(p4) for particle, p4 in sample_phase_space.items()}
sampled_c_selector = np.full(1, -1.0)
for name in ["D", "K", "D0", "pi"]:
    print(f"  sampled {name:>2}: {sampled_p4[name][0].tolist()}")


Random event setup: generate one TF-PWA phase-space event for Bp -> D K D0 pi.
  TF random seed: 2027980217
  sampled  D: [2.2106717882360427, 0.21574755988487204, 0.9951086504699128, 0.5955588946245497]
  sampled  K: [0.7447336804938349, 0.2724783776796438, -0.3984205883440598, 0.27915558925720263]
  sampled D0: [2.140192036853823, -0.45430460957247326, -0.514636953462229, -0.7947245771742772]
  sampled pi: [0.18374253665437412, -0.03392132783112655, -0.08205110846695951, -0.07998990641917515]


# Random Event Execution Flow


In [30]:
print("Random-event Step 1: Reconstruct intermediate four-vectors by summing daughters.")
sampled_p4["Dst"] = sampled_p4["D0"] + sampled_p4["pi"]
sampled_p4["Psi(4040)"] = sampled_p4["Dst"] + sampled_p4["D"]
sampled_p4["Bp"] = sampled_p4["Psi(4040)"] + sampled_p4["K"]
for name in ["Dst", "Psi(4040)", "Bp"]:
    print(f"  {name:>9}: {sampled_p4[name][0].tolist()}")


Random-event Step 1: Reconstruct intermediate four-vectors by summing daughters.
        Dst: [2.323934573508197, -0.4882259374035998, -0.5966880619291886, -0.8747144835934524]
  Psi(4040): [4.53460636174424, -0.2724783775187278, 0.39842058854072426, -0.2791555889689027]
         Bp: [5.279340042238075, 1.6091600274492635e-10, 1.9666446249289038e-10, 2.882999394770991e-10]


In [31]:
print("\nRandom-event Step 2: Compute invariant masses from the event kinematics.")
sampled_event_mass = {name: invariant_mass(vec) for name, vec in sampled_p4.items()}
for name in ["Bp", "Psi(4040)", "Dst", "D", "K", "D0", "pi"]:
    print(f"  m({name}) = {sampled_event_mass[name][0]:.12f} GeV")



Random-event Step 2: Compute invariant masses from the event kinematics.
  m(Bp) = 5.279340042238 GeV
  m(Psi(4040)) = 4.500193727111 GeV
  m(Dst) = 2.010259999337 GeV
  m(D) = 1.869650000000 GeV
  m(K) = 0.493677000000 GeV
  m(D0) = 1.864830000000 GeV
  m(pi) = 0.139570390000 GeV


In [32]:
print("\nRandom-event Step 3: Compute helicity Euler angles for the sequential decay chain.")
sampled_chain = [
    ("Bp", ["Psi(4040)", "K"]),
    ("Psi(4040)", ["Dst", "D"]),
    ("Dst", ["D0", "pi"]),
]
sampled_angles = calculate_helicity_angles(sampled_p4, sampled_chain)
for core, outs in sampled_chain:
    for out in outs:
        a = sampled_angles[core][out]
        print(f"  {core:>9} -> {out:<9}: alpha={a['alpha'][0]: .12f}, beta={a['beta'][0]: .12f}, gamma={a['gamma'][0]: .12f}")



Random-event Step 3: Compute helicity Euler angles for the sequential decay chain.
         Bp -> Psi(4040): alpha= 2.170632498769, beta= 2.095137659398, gamma= 0.000000000000
         Bp -> K        : alpha=-0.970960154821, beta= 1.046454994192, gamma= 0.000000000000
  Psi(4040) -> Dst      : alpha= 0.707104902817, beta= 1.602298383590, gamma= 0.000000000000
  Psi(4040) -> D        : alpha=-2.434487750773, beta= 1.539294270000, gamma= 0.000000000000
        Dst -> D0       : alpha= 2.940554065581, beta= 2.565385575508, gamma= 0.000000000000
        Dst -> pi       : alpha=-0.201038588009, beta= 0.576207078082, gamma= 0.000000000000


In [33]:
print("\nRandom-event Step 4: Compute breakup momenta q^2 and nominal q0^2 for barrier factors.")
sampled_q2_bp = get_relative_p2(sampled_event_mass["Bp"], sampled_event_mass["Psi(4040)"], sampled_event_mass["K"])
sampled_q02_bp = get_relative_p2(nominal_mass["Bp"], nominal_mass["Psi(4040)"], nominal_mass["K"])
sampled_q2_psi = get_relative_p2(sampled_event_mass["Psi(4040)"], sampled_event_mass["Dst"], sampled_event_mass["D"])
sampled_q02_psi = get_relative_p2(nominal_mass["Psi(4040)"], nominal_mass["Dst"], nominal_mass["D"])
sampled_q2_dst = get_relative_p2(sampled_event_mass["Dst"], sampled_event_mass["D0"], sampled_event_mass["pi"])
print(f"  Bp -> Psi K:      q2={sampled_q2_bp[0]:.12f}, q02={sampled_q02_bp:.12f}")
print(f"  Psi -> Dst D:     q2={sampled_q2_psi[0]:.12f}, q02={sampled_q02_psi:.12f}")
print(f"  Dst -> D0 pi:     q2={sampled_q2_dst[0]:.12f}; no barrier factor in this config")



Random-event Step 4: Compute breakup momenta q^2 and nominal q0^2 for barrier factors.
  Bp -> Psi K:      q2=0.310911274520, q02=1.005576573382
  Psi -> Dst D:     q2=1.298241821391, q02=0.314573138441
  Dst -> D0 pi:     q2=0.001549349855; no barrier factor in this config


In [34]:
print("\nRandom-event Step 5: Build the three helicity-decay tensors.")
spin0 = (0,)
spin1 = (-1, 0, 1)
sampled_amp_bp = helicity_decay_amp(
    core_j=0,
    out_js=(1, 0),
    core_spins=spin0,
    out_spins=(spin1, spin0),
    ls_list=((1, 1),),
    g_ls=g_bp_to_psi_k,
    angle=sampled_angles["Bp"]["Psi(4040)"],
    mass_core=sampled_event_mass["Bp"],
    q2=sampled_q2_bp,
    q02=sampled_q02_bp,
    has_barrier=True,
    barrier_norm=True,
)
sampled_amp_psi = helicity_decay_amp(
    core_j=1,
    out_js=(1, 0),
    core_spins=spin1,
    out_spins=(spin1, spin0),
    ls_list=((1, 1),),
    g_ls=g_psi_to_dst_d,
    angle=sampled_angles["Psi(4040)"]["Dst"],
    mass_core=sampled_event_mass["Psi(4040)"],
    q2=sampled_q2_psi,
    q02=sampled_q02_psi,
    has_barrier=True,
    barrier_norm=True,
)
sampled_amp_dst = helicity_decay_amp(
    core_j=1,
    out_js=(0, 0),
    core_spins=spin1,
    out_spins=(spin0, spin0),
    ls_list=((1, 0),),
    g_ls=g_dst_to_d0_pi,
    angle=sampled_angles["Dst"]["D0"],
    mass_core=sampled_event_mass["Dst"],
    q2=sampled_q2_dst,
    q02=0.0,
    has_barrier=False,
    barrier_norm=False,
)
print(f"  Bp -> Psi K tensor size: {sampled_amp_bp.shape}")
print(f"  Psi -> Dst D tensor size: {sampled_amp_psi.shape}")
print(f"  Dst -> D0 pi tensor size: {sampled_amp_dst.shape}")



Random-event Step 5: Build the three helicity-decay tensors.
  Bp -> Psi K tensor size: (1, 1, 3, 1)
  Psi -> Dst D tensor size: (1, 3, 3, 1)
  Dst -> D0 pi tensor size: (1, 3, 1, 1)


In [35]:
print("\nRandom-event Step 6: Compute particle factors.")
sampled_q_psi = np.sqrt(sampled_q2_psi)
sampled_q0_psi = np.sqrt(sampled_q02_psi)
sampled_psi_factor = bwr(sampled_event_mass["Psi(4040)"], nominal_mass["Psi(4040)"], psi_width, sampled_q_psi, sampled_q0_psi, l=1, d=3.0)
sampled_dst_factor = np.ones_like(sampled_psi_factor, dtype=complex)
print(f"  Psi(4040) factor: {format_complex(sampled_psi_factor)}")
print(f"  Dst model-one factor: {format_complex(sampled_dst_factor)}")



Random-event Step 6: Compute particle factors.
  Psi(4040) factor: -0.2453886047717168 + 0.04575930026074578j
  Dst model-one factor: 1 + 0j


In [36]:
print("\nRandom-event Step 7: Contract the tensors exactly like DecayChain.get_amp.")
sampled_particle_factor = g_total * sampled_psi_factor * sampled_dst_factor
sampled_amplitude_tensor = np.einsum("...agd,...gfb,...fce,...->...abcde", sampled_amp_bp, sampled_amp_psi, sampled_amp_dst, sampled_particle_factor)
sampled_amplitude = sampled_amplitude_tensor.reshape(-1)[0]
print(f"  Isolated sampled amplitude: {format_complex(sampled_amplitude)}")



Random-event Step 7: Contract the tensors exactly like DecayChain.get_amp.
  Isolated sampled amplitude: 0.003553312493232194 + 0.019054976585386j


In [37]:
print("\nRandom-event Step 8: Compare the isolated sampled amplitude with live TF-PWA.")
import os
import sys
repo_root = analysis_dir.parent
if str(repo_root / "tf-pwa") not in sys.path:
    sys.path.insert(0, str(repo_root / "tf-pwa"))
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))
os.chdir(analysis_dir)

import extra_amp
from tf_pwa.config_loader import ConfigLoader

sample_config = ConfigLoader("config_a.yml")
with open("final_params_full.json", "r", encoding="utf-8") as f:
    sample_params_dict = json.load(f)["value"]
sample_particles = list(sample_config.get_decay().outs)
sample_particle_map = {p.name: p for p in sample_particles}
sampled_p4_tfpwa = {
    sample_particle_map["D"]: tf.constant(sampled_p4["D"], dtype=tf.float64),
    sample_particle_map["D0"]: tf.constant(sampled_p4["D0"], dtype=tf.float64),
    sample_particle_map["K"]: tf.constant(sampled_p4["K"], dtype=tf.float64),
    sample_particle_map["pi"]: tf.constant(sampled_p4["pi"], dtype=tf.float64),
}
sampled_phsp_variables = sample_config.data.cal_angle(sampled_p4_tfpwa)
sampled_phsp_variables["c"] = sampled_c_selector
sample_amp_model = sample_config.get_amplitude()
sample_dg = sample_amp_model.decay_group
sample_chain = sample_dg.chains[5]
sample_p_unit = sample_params_dict.copy()
for key in sample_p_unit:
    if ("total" in key or "g_ls" in key) and (key.endswith("r") or key.endswith("i")):
        sample_p_unit[key] = 0.0
for d_idx, decay in enumerate(sample_chain.chain):
    prefix = f"{decay.core.name.replace('(1+)', '(1.)')}->{'.'.join([p.name.replace('(1+)', '(1.)') for p in decay.outs])}"
    for key in sample_p_unit:
        if prefix in key and ("total" in key or "g_ls" in key) and key.endswith("_0r"):
            sample_p_unit[key] = 1.0
sample_config.set_params(sample_p_unit)
sample_dg.set_used_chains([5])
sampled_tfpwa_live = sample_dg.get_amp(sampled_phsp_variables).numpy().reshape(-1)[0]
print(f"  Isolated sampled amplitude: {format_complex(sampled_amplitude)}")
print(f"  Live TF-PWA amplitude:    {format_complex(sampled_tfpwa_live)}")
print(f"  Sampled-event difference: {sampled_amplitude - sampled_tfpwa_live}")
assert np.abs(sampled_amplitude - sampled_tfpwa_live) < 5e-10, "Sampled-event isolated amplitude does not match live TF-PWA."
print("  PASS: sampled event also matches live TF-PWA.")



Random-event Step 8: Compare the isolated sampled amplitude with live TF-PWA.
  Isolated sampled amplitude: 0.003553312493232194 + 0.019054976585386j
  Live TF-PWA amplitude:    0.003553312493232211 + 0.01905497658538605j
  Sampled-event difference: (-1.6479873021779667e-17-4.163336342344337e-17j)
  PASS: sampled event also matches live TF-PWA.


In [38]:
# Validate the isolated calculation against TF-PWA for 1,000 configured phase-space events.
batch_n_events = 1000
batch_seed = int(np.random.default_rng().integers(0, 2**31 - 1))
print(f"\nBatch validation: {batch_n_events} TF-PWA phase-space events (seed {batch_seed}).")
np.random.seed(batch_seed)
tf.random.set_seed(batch_seed)

batch_config = ConfigLoader("config_a.yml")
batch_phase_space = batch_config.generate_phsp_p(batch_n_events)
batch_p4 = {particle.name: np.asarray(p4) for particle, p4 in batch_phase_space.items()}
batch_p4["Dst"] = batch_p4["D0"] + batch_p4["pi"]
batch_p4["Psi(4040)"] = batch_p4["Dst"] + batch_p4["D"]
batch_p4["Bp"] = batch_p4["Psi(4040)"] + batch_p4["K"]
batch_event_mass = {name: invariant_mass(vec) for name, vec in batch_p4.items()}

batch_angles = calculate_helicity_angles(batch_p4, sampled_chain)
batch_q2_bp = get_relative_p2(batch_event_mass["Bp"], batch_event_mass["Psi(4040)"], batch_event_mass["K"])
batch_q02_bp = get_relative_p2(nominal_mass["Bp"], nominal_mass["Psi(4040)"], nominal_mass["K"])
batch_q2_psi = get_relative_p2(batch_event_mass["Psi(4040)"], batch_event_mass["Dst"], batch_event_mass["D"])
batch_q02_psi = get_relative_p2(nominal_mass["Psi(4040)"], nominal_mass["Dst"], nominal_mass["D"])
batch_q2_dst = get_relative_p2(batch_event_mass["Dst"], batch_event_mass["D0"], batch_event_mass["pi"])

batch_amp_bp = helicity_decay_amp(
    core_j=0, out_js=(1, 0), core_spins=spin0, out_spins=(spin1, spin0),
    ls_list=((1, 1),), g_ls=g_bp_to_psi_k, angle=batch_angles["Bp"]["Psi(4040)"],
    mass_core=batch_event_mass["Bp"], q2=batch_q2_bp, q02=batch_q02_bp,
    has_barrier=True, barrier_norm=True,
)
batch_amp_psi = helicity_decay_amp(
    core_j=1, out_js=(1, 0), core_spins=spin1, out_spins=(spin1, spin0),
    ls_list=((1, 1),), g_ls=g_psi_to_dst_d, angle=batch_angles["Psi(4040)"]["Dst"],
    mass_core=batch_event_mass["Psi(4040)"], q2=batch_q2_psi, q02=batch_q02_psi,
    has_barrier=True, barrier_norm=True,
)
batch_amp_dst = helicity_decay_amp(
    core_j=1, out_js=(0, 0), core_spins=spin1, out_spins=(spin0, spin0),
    ls_list=((1, 0),), g_ls=g_dst_to_d0_pi, angle=batch_angles["Dst"]["D0"],
    mass_core=batch_event_mass["Dst"], q2=batch_q2_dst, q02=0.0,
    has_barrier=False, barrier_norm=False,
)
batch_psi_factor = bwr(
    batch_event_mass["Psi(4040)"], nominal_mass["Psi(4040)"], psi_width,
    np.sqrt(batch_q2_psi), np.sqrt(batch_q02_psi), l=1, d=3.0,
)
batch_particle_factor = g_total * batch_psi_factor * np.ones_like(batch_psi_factor, dtype=complex)
batch_amplitude_tensor = np.einsum(
    "...agd,...gfb,...fce,...->...abcde",
    batch_amp_bp, batch_amp_psi, batch_amp_dst, batch_particle_factor,
)
batch_isolated_amplitude = batch_amplitude_tensor.reshape(batch_n_events, -1)[:, 0]

batch_particles = list(batch_config.get_decay().outs)
batch_particle_map = {particle.name: particle for particle in batch_particles}
batch_p4_tfpwa = {
    batch_particle_map[name]: tf.constant(batch_p4[name], dtype=tf.float64)
    for name in ["D", "D0", "K", "pi"]
}
batch_phsp_variables = batch_config.data.cal_angle(batch_p4_tfpwa)
batch_phsp_variables["c"] = np.full(batch_n_events, -1.0)
batch_amp_model = batch_config.get_amplitude()
batch_dg = batch_amp_model.decay_group
batch_chain = batch_dg.chains[5]
batch_p_unit = params.copy()
for key in batch_p_unit:
    if ("total" in key or "g_ls" in key) and (key.endswith("r") or key.endswith("i")):
        batch_p_unit[key] = 0.0
for decay in batch_chain.chain:
    prefix = f"{decay.core.name.replace('(1+)', '(1.)')}->{'.'.join([p.name.replace('(1+)', '(1.)') for p in decay.outs])}"
    for key in batch_p_unit:
        if prefix in key and ("total" in key or "g_ls" in key) and key.endswith("_0r"):
            batch_p_unit[key] = 1.0
batch_config.set_params(batch_p_unit)
batch_dg.set_used_chains([5])
batch_tfpwa_amplitude = batch_dg.get_amp(batch_phsp_variables).numpy().reshape(-1)

batch_difference = batch_isolated_amplitude - batch_tfpwa_amplitude
batch_abs_difference = np.abs(batch_difference)
batch_worst_event = int(np.argmax(batch_abs_difference))
print(f"  Mean |isolated - TF-PWA|: {np.mean(batch_abs_difference):.6e}")
print(f"  Max  |isolated - TF-PWA|: {batch_abs_difference[batch_worst_event]:.6e} (event {batch_worst_event})")
print(f"  Isolated amplitude at worst event: {format_complex(batch_isolated_amplitude[batch_worst_event])}")
print(f"  TF-PWA amplitude at worst event:   {format_complex(batch_tfpwa_amplitude[batch_worst_event])}")
assert np.all(batch_abs_difference < 5e-10), "At least one of the 1,000 isolated amplitudes does not match TF-PWA."
print("  PASS: all 1,000 isolated amplitudes match TF-PWA.")



Batch validation: 1000 TF-PWA phase-space events (seed 1529459598).
  Mean |isolated - TF-PWA|: 1.354989e-15
  Max  |isolated - TF-PWA|: 5.392713e-14 (event 304)
  Isolated amplitude at worst event: -1.346663346528924 - 0.8022135061699657j
  TF-PWA amplitude at worst event:   -1.346663346528873 - 0.802213506169983j
  PASS: all 1,000 isolated amplitudes match TF-PWA.


# Full configured model validation

The following cells evaluate each active decay chain independently and then form the coherent all-component amplitude.


In [39]:
# Evaluate all configured components with the isolated NumPy implementation.
full_single_p4 = {name: sampled_p4[name] for name in ["D", "K", "D0", "pi"]}
full_single_isolated_components = [
    isolated_component_amplitude(full_single_p4, spec)[0]
    for spec in component_specs
]
full_single_isolated = sum(full_single_isolated_components)

print("Isolated component amplitudes for the sampled event:")
for spec, amplitude_value in zip(component_specs, full_single_isolated_components):
    print(f"  {spec['name']:<18}: {format_complex(amplitude_value)}")
print(f"  {'coherent sum':<18}: {format_complex(full_single_isolated)}")


Isolated component amplitudes for the sampled event:
  X(3872)           : 0.0335629463247878 - 0.03738220457978235j
  X(3915)(0-)       : -0.01363931165642394 + 0.04966306338456852j
  chi(c2)(3930)     : -0.0001273905447579018 + 0.00021176430464842j
  X(3940)(1.)       : 0.004428210607554556 - 0.04010140282492667j
  X(3993)           : -0.01090978724946114 - 0.005472665698882291j
  Psi(4040)         : -0.004649574099281828 - 0.0003643483655549409j
  X(4300)           : 0.02270649618291962 + 0.003163766207206873j
  NR(0-)SPp         : 0.2496395408852381 - 0.2413937640737871j
  NR(1.)PSp         : 0.2060924856172358 + 0.01061032041402082j
  NR(0-)SPm         : -0.08510532186722949 - 0.000469519997731027j
  NR(1-)PPm         : 0.02000987264533866 + 0.05150995303536354j
  X0(2900)          : 0.08517942686422363 + 0.03336180946554181j
  X1(2900)          : -0.0145616406837116 + 0.02739643027941148j
  coherent sum      : 0.4926259530264322 - 0.1492667984499029j


In [40]:
# Compare every isolated component and the coherent sum with the original TF-PWA model.
full_single_config = ConfigLoader("config_a.yml")
full_single_particles = {particle.name: particle for particle in full_single_config.get_decay().outs}
full_single_tfpwa_p4 = {
    full_single_particles[name]: tf.constant(full_single_p4[name], dtype=tf.float64)
    for name in ["D", "D0", "K", "pi"]
}
full_single_data = full_single_config.data.cal_angle(full_single_tfpwa_p4)
full_single_data["c"] = np.full(1, -1.0)
full_single_model = full_single_config.get_amplitude()
full_single_group = full_single_model.decay_group
full_single_config.set_params(params)

full_single_live_components = []
print("\nOne-event isolated versus TF-PWA comparison:")
for chain_index, (spec, isolated_value) in enumerate(zip(component_specs, full_single_isolated_components)):
    full_single_group.set_used_chains([chain_index])
    live_value = full_single_group.get_amp(full_single_data).numpy().reshape(-1)[0]
    full_single_live_components.append(live_value)
    difference = isolated_value - live_value
    print(f"  {spec['name']:<18}: |difference| = {abs(difference):.6e}")

full_single_group.set_used_chains(list(range(len(component_specs))))
full_single_live = full_single_group.get_amp(full_single_data).numpy().reshape(-1)[0]
full_single_difference = full_single_isolated - full_single_live
print(f"  {'coherent sum':<18}: |difference| = {abs(full_single_difference):.6e}")
assert max(abs(np.asarray(full_single_isolated_components) - np.asarray(full_single_live_components))) < 1e-9
assert abs(full_single_difference) < 1e-9
print("  PASS: every component and the coherent sum match TF-PWA for the sampled event.")



One-event isolated versus TF-PWA comparison:
  X(3872)           : |difference| = 7.376145e-17
  X(3915)(0-)       : |difference| = 9.728381e-17
  chi(c2)(3930)     : |difference| = 2.790636e-19
  X(3940)(1.)       : |difference| = 7.773903e-17
  X(3993)           : |difference| = 3.648079e-17
  Psi(4040)         : |difference| = 1.127622e-17
  X(4300)           : |difference| = 1.353084e-16
  NR(0-)SPp         : |difference| = 1.344516e-10
  NR(1.)PSp         : |difference| = 9.026717e-11
  NR(0-)SPm         : |difference| = 3.295117e-11
  NR(1-)PPm         : |difference| = 4.556683e-11
  X0(2900)          : |difference| = 3.103168e-17
  X1(2900)          : |difference| = 2.501854e-17
  coherent sum      : |difference| = 1.765905e-10
  PASS: every component and the coherent sum match TF-PWA for the sampled event.


## Validation on 50,000 generated events

The configured generator supplies one common event sample to the isolated implementation and the original TF-PWA model.


In [41]:
# Generate 50,000 configured events and validate every component plus the coherent sum.
full_batch_n_events = 50000
full_batch_seed = int(np.random.default_rng().integers(0, 2**31 - 1))
print(f"Full-model batch validation: {full_batch_n_events} events (seed {full_batch_seed}).")
np.random.seed(full_batch_seed)
tf.random.set_seed(full_batch_seed)

full_batch_config = ConfigLoader("config_a.yml")
full_batch_generated = full_batch_config.generate_phsp_p(full_batch_n_events)
full_batch_p4 = {particle.name: np.asarray(p4) for particle, p4 in full_batch_generated.items()}
full_batch_isolated_components = np.stack([
    isolated_component_amplitude(full_batch_p4, spec)
    for spec in component_specs
])
full_batch_isolated = np.sum(full_batch_isolated_components, axis=0)

full_batch_particles = {particle.name: particle for particle in full_batch_config.get_decay().outs}
full_batch_tfpwa_p4 = {
    full_batch_particles[name]: tf.constant(full_batch_p4[name], dtype=tf.float64)
    for name in ["D", "D0", "K", "pi"]
}
full_batch_data = full_batch_config.data.cal_angle(full_batch_tfpwa_p4)
full_batch_data["c"] = np.full(full_batch_n_events, -1.0)
full_batch_model = full_batch_config.get_amplitude()
full_batch_group = full_batch_model.decay_group
full_batch_config.set_params(params)

full_batch_live_components = []
print("\nMaximum absolute component differences:")
for chain_index, spec in enumerate(component_specs):
    full_batch_group.set_used_chains([chain_index])
    live_component = full_batch_group.get_amp(full_batch_data).numpy().reshape(-1)
    full_batch_live_components.append(live_component)
    component_difference = np.abs(full_batch_isolated_components[chain_index] - live_component)
    print(f"  {spec['name']:<18}: mean={np.mean(component_difference):.6e}, max={np.max(component_difference):.6e}")

full_batch_live_components = np.stack(full_batch_live_components)
full_batch_group.set_used_chains(list(range(len(component_specs))))
full_batch_live = full_batch_group.get_amp(full_batch_data).numpy().reshape(-1)
full_batch_difference = np.abs(full_batch_isolated - full_batch_live)
full_batch_component_difference = np.abs(full_batch_isolated_components - full_batch_live_components)
print(f"  {'coherent sum':<18}: mean={np.mean(full_batch_difference):.6e}, max={np.max(full_batch_difference):.6e}")
assert np.max(full_batch_component_difference) < 1e-9, "At least one isolated component does not match TF-PWA."
assert np.max(full_batch_difference) < 1e-9, "The isolated coherent sum does not match TF-PWA."
print(f"  PASS: all 13 components and the coherent sum match TF-PWA for all {full_batch_n_events:,} events.")


Full-model batch validation: 50000 events (seed 266331871).

Maximum absolute component differences:
  X(3872)           : mean=4.601518e-16, max=2.811694e-13
  X(3915)(0-)       : mean=2.830022e-16, max=3.829263e-14
  chi(c2)(3930)     : mean=2.619487e-16, max=9.020403e-14
  X(3940)(1.)       : mean=3.253244e-16, max=1.279710e-14
  X(3993)           : mean=5.788148e-16, max=7.056447e-14
  Psi(4040)         : mean=2.785542e-16, max=2.103633e-14
  X(4300)           : mean=2.651768e-16, max=1.809185e-14
  NR(0-)SPp         : mean=6.705772e-11, max=2.167725e-10
  NR(1.)PSp         : mean=8.244153e-11, max=1.799087e-10
  NR(0-)SPm         : mean=1.875390e-11, max=3.972358e-11
  NR(1-)PPm         : mean=1.603910e-10, max=4.374780e-10
  X0(2900)          : mean=2.500450e-16, max=1.443557e-14
  X1(2900)          : mean=2.096046e-16, max=8.169977e-15
  coherent sum      : mean=2.057548e-10, max=4.456477e-10
  PASS: all 13 components and the coherent sum match TF-PWA for all 50,000 events.


In [ ]:
# Compare the full TF-PWA and isolated amplitudes across the sampled phase space.
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import FuncFormatter

# These invariant-mass combinations expose the DstD and DK resonance topologies.
full_batch_p_dst = full_batch_p4["D0"] + full_batch_p4["pi"]
dalitz_m2_dstd = invariant_mass(full_batch_p_dst + full_batch_p4["D"]) ** 2
dalitz_m2_dk = invariant_mass(full_batch_p4["D"] + full_batch_p4["K"]) ** 2

tfpwa_intensity = np.abs(full_batch_live) ** 2
isolated_intensity = np.abs(full_batch_isolated) ** 2
reference_magnitude = np.abs(full_batch_live)
denominator_threshold = np.finfo(float).eps * max(1.0, float(np.max(reference_magnitude)))
relative_amplitude_difference = np.full(full_batch_n_events, np.nan, dtype=float)
valid_denominator = reference_magnitude > denominator_threshold
relative_amplitude_difference[valid_denominator] = (
    np.abs(full_batch_isolated[valid_denominator] - full_batch_live[valid_denominator])
    / reference_magnitude[valid_denominator]
)

# The 50,000-event sample supports four times the previous resolution per axis.
dalitz_nbins = 100
x_edges = np.linspace(np.min(dalitz_m2_dstd), np.max(dalitz_m2_dstd), dalitz_nbins + 1)
y_edges = np.linspace(np.min(dalitz_m2_dk), np.max(dalitz_m2_dk), dalitz_nbins + 1)
counts = np.histogram2d(dalitz_m2_dstd, dalitz_m2_dk, bins=(x_edges, y_edges))[0]
tfpwa_sums = np.histogram2d(
    dalitz_m2_dstd, dalitz_m2_dk, bins=(x_edges, y_edges), weights=tfpwa_intensity
)[0]
isolated_sums = np.histogram2d(
    dalitz_m2_dstd, dalitz_m2_dk, bins=(x_edges, y_edges), weights=isolated_intensity
)[0]
tfpwa_grid = np.divide(
    tfpwa_sums, counts, out=np.full_like(tfpwa_sums, np.nan), where=counts > 0
)
isolated_grid = np.divide(
    isolated_sums, counts, out=np.full_like(isolated_sums, np.nan), where=counts > 0
)

valid_relative = np.isfinite(relative_amplitude_difference)
relative_counts = np.histogram2d(
    dalitz_m2_dstd[valid_relative], dalitz_m2_dk[valid_relative], bins=(x_edges, y_edges)
)[0]
relative_sums = np.histogram2d(
    dalitz_m2_dstd[valid_relative],
    dalitz_m2_dk[valid_relative],
    bins=(x_edges, y_edges),
    weights=relative_amplitude_difference[valid_relative],
)[0]
relative_grid = np.divide(
    relative_sums,
    relative_counts,
    out=np.full_like(relative_sums, np.nan),
    where=relative_counts > 0,
)

intensity_positive = np.concatenate([
    tfpwa_grid[np.isfinite(tfpwa_grid) & (tfpwa_grid > 0)],
    isolated_grid[np.isfinite(isolated_grid) & (isolated_grid > 0)],
])
intensity_vmin = max(float(np.quantile(intensity_positive, 0.02)), np.finfo(float).tiny)
intensity_vmax = max(float(np.quantile(intensity_positive, 0.98)), 10.0 * intensity_vmin)
intensity_norm = LogNorm(vmin=intensity_vmin, vmax=intensity_vmax)

mean_relative_difference = float(np.nanmean(relative_amplitude_difference))
if np.isfinite(mean_relative_difference) and mean_relative_difference > 0:
    relative_scale_exponent = int(np.floor(np.log10(mean_relative_difference)))
else:
    relative_scale_exponent = 0
relative_scale = 10.0 ** relative_scale_exponent
scaled_relative_grid = relative_grid / relative_scale

relative_positive = scaled_relative_grid[
    np.isfinite(scaled_relative_grid) & (scaled_relative_grid > 0)
]
if relative_positive.size:
    relative_vmin = max(float(np.min(relative_positive)), np.finfo(float).tiny)
    relative_vmax = max(float(np.max(relative_positive)), 10.0 * relative_vmin)
    relative_norm = LogNorm(vmin=relative_vmin, vmax=relative_vmax)
    relative_plot_grid = np.ma.masked_less_equal(scaled_relative_grid, 0.0)
    relative_ticks = np.geomspace(relative_vmin, relative_vmax, 5)
else:
    relative_norm = Normalize(vmin=0.0, vmax=1.0)
    relative_plot_grid = np.ma.masked_invalid(scaled_relative_grid)
    relative_ticks = np.linspace(0.0, 1.0, 5)

intensity_cmap = plt.get_cmap("viridis").copy()
intensity_cmap.set_bad("white")
relative_cmap = plt.get_cmap("magma").copy()
relative_cmap.set_bad("white")

comparison_figure, comparison_axes = plt.subplots(
    1, 3, figsize=(18.0, 5.2), sharex=True, sharey=True, constrained_layout=True
)
plot_extent = (x_edges[0], x_edges[-1], y_edges[0], y_edges[-1])
tfpwa_image = comparison_axes[0].imshow(
    tfpwa_grid.T, origin="lower", extent=plot_extent, aspect="auto",
    interpolation="nearest", cmap=intensity_cmap, norm=intensity_norm, rasterized=True
)
comparison_axes[1].imshow(
    isolated_grid.T, origin="lower", extent=plot_extent, aspect="auto",
    interpolation="nearest", cmap=intensity_cmap, norm=intensity_norm, rasterized=True
)
relative_image = comparison_axes[2].imshow(
    relative_plot_grid.T, origin="lower", extent=plot_extent, aspect="auto",
    interpolation="nearest", cmap=relative_cmap, norm=relative_norm, rasterized=True
)
comparison_axes[0].set_title(r"Original TF-PWA: mean $|A|^2$")
comparison_axes[1].set_title(r"Isolated reimplementation: mean $|A|^2$")
comparison_axes[2].set_title(
    rf"Mean $|A_{{\mathrm{{iso}}}}-A_{{\mathrm{{TF}}}}|/|A_{{\mathrm{{TF}}}}|\,/\,10^{{{relative_scale_exponent}}}$"
)
for axis in comparison_axes:
    axis.set_xlabel(r"$m^2(D^{*}D)\;[\mathrm{GeV}^2]$")
comparison_axes[0].set_ylabel(r"$m^2(DK)\;[\mathrm{GeV}^2]$")
comparison_figure.colorbar(tfpwa_image, ax=comparison_axes[:2], pad=0.015)
relative_colorbar = comparison_figure.colorbar(
    relative_image, ax=comparison_axes[2], pad=0.015, ticks=relative_ticks
)
relative_colorbar.formatter = FuncFormatter(lambda value, _: f"{value:.2g}")
relative_colorbar.update_ticks()

plots_dir = Path("../notebooks/Plots")
plots_dir.mkdir(parents=True, exist_ok=True)
comparison_plot_path = plots_dir / "all_components_amplitude_comparison_dalitz.pdf"
comparison_figure.savefig(comparison_plot_path, bbox_inches="tight", dpi=300)
plt.show()

print(f"Saved amplitude comparison to {comparison_plot_path.resolve()}")
print(f"Relative-difference color scale: values divided by 10^{relative_scale_exponent}")
print(
    "Relative complex-amplitude difference: "
    f"median={np.nanmedian(relative_amplitude_difference):.6e}, "
    f"max={np.nanmax(relative_amplitude_difference):.6e}"
)
